In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

In [2]:
doc_sent = pd.read_csv("../data/output/DOC_SENT.csv", sep="|")
lib = pd.read_csv("../data/archive/key_english.csv")

In [3]:
lib.columns = ["book_number", "book_name", "testament", "genre"]

doc_sent = doc_sent.merge(
    lib[["book_number", "book_name", "testament"]],
    on="book_number",
    how="left"
)

In [4]:
emotion = "joy"

In [5]:
doc_sent = doc_sent.sort_values(["book_number", "chapter", "verse"]).reset_index(drop=True)

doc_sent["canon_position"] = np.arange(len(doc_sent))

In [6]:
threshold = doc_sent[emotion].mean()

dispersion_df = doc_sent.copy()
dispersion_df["emotion_flag"] = dispersion_df[emotion] > threshold

In [7]:
fig = px.scatter(
    doc_sent,
    x="canon_position",
    y="book_name",
    color="joy",
    hover_data=["book_name", "chapter", "verse", "joy"],
    color_continuous_scale="Plasma",
    title="Joy Dispersion Across Canonical Order",
    height=900
)

fig.update_layout(
    xaxis_title="Canonical Position",
    yaxis_title="",
    yaxis=dict(
        showticklabels=False,
        categoryorder="array",
        categoryarray=lib.sort_values("book_number")["book_name"].tolist()
    ),
    coloraxis_colorbar=dict(
        title="Joy Intensity"
    )
)

fig.update_traces(
    marker=dict(
        size=3,
        opacity=0.75
    )
)

# Clamp color range to make variation visible
fig.update_coloraxes(
    cmin=0,
    cmax=doc_sent["joy"].quantile(0.98)
)

fig.show()

In [8]:
# Compute book-level average joy
book_joy = (
    doc_sent
    .groupby("book_name")["joy"]
    .mean()
    .reset_index()
)

# Merge canonical order
book_joy = (
    book_joy
    .merge(lib[["book_name", "book_number"]], on="book_name")
    .sort_values("book_number")
)

fig = px.scatter(
    book_joy,
    x="book_number",
    y="joy",
    hover_data=["book_name"],
    color="joy",
    color_continuous_scale="Plasma",
    title="Average Joy by Book Across Canonical Order",
    labels={
        "book_number": "Canonical Order",
        "joy": "Average Joy"
    }
)

fig.update_traces(marker=dict(size=9))

fig.update_layout(
    coloraxis_colorbar=dict(title="Average Joy")
)

fig.show()

In [9]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Aggregate book-level averages for all emotions
book_emotions = (
    doc_sent
    .groupby("book_name")[[
        "joy", "trust", "anticipation", "surprise",
        "anger", "fear", "disgust", "sadness"
    ]]
    .mean()
    .reset_index()
)

# Merge canonical order
book_emotions = (
    book_emotions
    .merge(lib[["book_name", "book_number"]], on="book_name")
    .sort_values("book_number")
)

positive_emotions = ["joy", "trust", "anticipation", "surprise"]
negative_emotions = ["anger", "fear", "disgust", "sadness"]

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=positive_emotions + negative_emotions,
    shared_xaxes=True
)

# ---- Top Row: Positive Emotions ----
for i, emotion in enumerate(positive_emotions):
    fig.add_trace(
        go.Scatter(
            x=book_emotions["book_number"],
            y=book_emotions[emotion],
            mode="markers",
            marker=dict(size=7),
            text=book_emotions["book_name"],
            hovertemplate="Book: %{text}<br>" + emotion.capitalize() + ": %{y:.4f}<extra></extra>"
        ),
        row=1, col=i+1
    )

# ---- Bottom Row: Negative Emotions ----
for i, emotion in enumerate(negative_emotions):
    fig.add_trace(
        go.Scatter(
            x=book_emotions["book_number"],
            y=book_emotions[emotion],
            mode="markers",
            marker=dict(size=7),
            text=book_emotions["book_name"],
            hovertemplate="Book: %{text}<br>" + emotion.capitalize() + ": %{y:.4f}<extra></extra>"
        ),
        row=2, col=i+1
    )

fig.update_layout(
    height=800,
    width=1400,
    title_text="Book-Level Emotional Profiles Across Canonical Order",
    showlegend=False
)

fig.update_xaxes(title_text="Canonical Order")
fig.update_yaxes(title_text="Average Emotion Score")

fig.show()

In [10]:
from scipy.stats import pearsonr

emotions = [
    "joy", "trust", "anticipation", "surprise",
    "anger", "fear", "disgust", "sadness"
]

r_values = []

for emotion in emotions:
    r, p = pearsonr(book_emotions["book_number"], book_emotions[emotion])
    r_values.append({
        "emotion": emotion,
        "r_value": r,
        "p_value": p
    })

r_df = pd.DataFrame(r_values).sort_values("r_value", ascending=False)

r_df

,emotion,r_value,p_value
0,joy,0.710737,2.303919e-11
2,anticipation,0.690788,1.373786e-10
1,trust,0.647528,4.205583e-09
5,fear,0.495725,2.313188e-05
3,surprise,0.346183,4.409480e-03
4,anger,0.263528,3.252049e-02
7,sadness,0.182241,1.430463e-01
6,disgust,0.049179,6.949582e-01


Across canonical order, the strongest and most structurally meaningful pattern appears in joy. While the correlation is strongly positive, the visual pattern is not simply a smooth increase from beginning to end. Instead, the Old Testament books cluster at relatively low joy levels, and the Gospels and Acts remain close to that same range. The major shift occurs after Acts, where the remainder of the New Testament shows a clear upward jump in average joy. In other words, the emotional break is not between the Old and New Testaments as a whole, but between the narrative material of the Gospels and Acts and the later epistolary material. The rest of the New Testament sustains noticeably higher joy levels, forming a distinct upper cluster in the plot. This clustering structure explains the high correlation value: the relationship is driven by a structural step change rather than gradual drift.

Anticipation and trust follow a similar though slightly smoother pattern. Both emotions show strong positive correlations with canonical order, indicating that later books tend to carry higher levels of these forward-looking and affirmational tones. However, unlike joy, the increase appears more gradual across canonical progression, with less sharply defined clustering. The New Testament still trends higher overall, but the separation is less step-like and more continuous.

Fear also increases across canonical order, though more moderately. This suggests that later books are not simply more positive in tone, but may exhibit greater emotional intensity overall. The rise in fear in later canonical material likely reflects apocalyptic and eschatological themes, which intensify emotional language even while positive emotions increase.

Anger, sadness, and disgust do not show strong structural trends. Their correlations are weak or non-significant, and their plots appear more dispersed without clear clustering by canonical division. This indicates that not all emotional dimensions track canonical order in the same way. The strongest canonical structuring occurs in positive affect, especially joy, where the primary interpretive insight is the post-Acts jump into a distinctly higher emotional register in the later New Testament.

In [11]:
import os

graph_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs"

save_file = os.path.join(
    graph_path,
    "25_riff_2_book_level_emotional_profiles.html"
)

fig.write_html(save_file)

fig.show()

print("Saved to:", save_file)

Saved to: /Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs/25_riff_2_book_level_emotional_profiles.html
